In [9]:
import json
import re
import pandas as pd
import yaml

# =========================
# Experiment Configuration
# =========================
CSV_PATH = "../result/iacgod/grok_secure_results_agg.csv"
QUERY_MODE = "from_csv"  # "from_csv" or "custom"
ERROR_SOURCE_COLUMN = "latest_error_cfn-lint"  # pick one: latest_error_cfn-lint or latest_error_yaml
CUSTOM_ERROR_TEXT = "[E3002] Invalid Property Resources/MyTopic/Properties/Foo"
CASE_INDEX = 3  # which error case to use from CSV
N_RESULTS_VECTOR = 20
TOP_K_GRAPH = 20
TOP_K_CONTEXT = 5
MAX_OPTIONAL_SHOWN = 20

def extract_error_from_row(row, column_name):
    value = row.get(column_name)
    if pd.isna(value):
        return ""
    parts = [e.strip() for e in str(value).split("|") if e.strip()]
    return parts[0] if parts else ""

# Load benchmark rows and build selectable error cases
df = pd.read_csv(CSV_PATH)
df_errors = df[
    (df["latest_error_cfn-lint"].notna()) | (df["latest_error_yaml"].notna())
].copy()

if ERROR_SOURCE_COLUMN not in ["latest_error_cfn-lint", "latest_error_yaml"]:
    raise ValueError(f"ERROR_SOURCE_COLUMN must be one of ['latest_error_cfn-lint', 'latest_error_yaml']; got {ERROR_SOURCE_COLUMN}")

cases = []
for _, row in df_errors.iterrows():
    err = extract_error_from_row(row, ERROR_SOURCE_COLUMN)
    if err:
        cases.append({
            "run_id": row.get("run_id", ""),
            "template": row.get("final_template", ""),
            "error": err,
        })

if QUERY_MODE == "from_csv":
    if not cases:
        raise ValueError("No CSV error cases were found.")
    if CASE_INDEX < 0 or CASE_INDEX >= len(cases):
        raise IndexError(f"CASE_INDEX={CASE_INDEX} out of bounds (0..{len(cases)-1})")
    selected_case = cases[CASE_INDEX]
    query_text = selected_case["error"]
    selected_template = selected_case["template"]
    selected_run_id = selected_case["run_id"]
else:
    selected_case = None
    query_text = CUSTOM_ERROR_TEXT
    selected_template = ""
    selected_run_id = "custom_query"

print(f"Loaded {len(cases)} error cases from CSV")
print(f"Query mode: {QUERY_MODE}")
print(f"Error source column: {ERROR_SOURCE_COLUMN if QUERY_MODE == 'from_csv' else 'custom'}")
print(f"Selected run_id: {selected_run_id}")
print(f"Query text length: {len(query_text)} chars")
print(f"Query preview: {query_text}")

Loaded 109 error cases from CSV
Query mode: from_csv
Error source column: latest_error_cfn-lint
Selected run_id: 20260412_014839_fab97a2b
Query text length: 82 chars
Query preview: [E3003] {'ColumnNumber': 7, 'LineNumber': 20}: 'SSEEnabled' is a required property


In [2]:
import os
import pickle
import networkx as nx

GRAPH_PATH = "../data/cfn_graph.pkl"

def load_graph(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing graph file: {path}")
    with open(path, "rb") as f:
        obj = pickle.load(f)
    return obj[0] if isinstance(obj, tuple) else obj

G = load_graph(GRAPH_PATH)
print(f"Graph loaded. Nodes={G.number_of_nodes()} Edges={G.number_of_edges()}")

# Build corpus for vector retrieval
corpus = []
corpus_labels = []

for node_id, data in G.nodes(data=True):
    ntype = data.get("ntype", "")
    if ntype in ("ResourceType", "Resource"):
        corpus.append(f"Resource Schema: {node_id}. {data.get('docs', '')}")
        corpus_labels.append(node_id)
    elif ntype == "Property":
        parts = str(node_id).rsplit("/", 1)
        rtype = parts[0] if len(parts) == 2 else node_id
        prop = parts[1] if len(parts) == 2 else data.get("name", "")
        ptype = data.get("primitive_type") or data.get("type") or "Any"
        required = data.get("required", False)
        corpus.append(f"{rtype} Property: {prop}. Type: {ptype}. Required: {required}.")
        corpus_labels.append(f"{rtype}/{prop}")

print(f"Vector corpus built with {len(corpus)} docs")

Graph loaded. Nodes=47058 Edges=45478
Vector corpus built with 38494 docs


In [10]:
# ==========================
# Retrieval and Schema Utils
# ==========================
def _cfn_tag_constructor(loader, tag_suffix, node):
    if isinstance(node, yaml.ScalarNode):
        return loader.construct_scalar(node)
    if isinstance(node, yaml.SequenceNode):
        return loader.construct_sequence(node)
    if isinstance(node, yaml.MappingNode):
        return loader.construct_mapping(node)
    return None

yaml.SafeLoader.add_multi_constructor("!", _cfn_tag_constructor)

def parse_template_resources(template_yaml):
    if not template_yaml or pd.isna(template_yaml):
        return {}
    try:
        tpl = yaml.safe_load(template_yaml) or {}
        resources = tpl.get("Resources", {})
        return {k: v.get("Type") for k, v in resources.items() if isinstance(v, dict)}
    except Exception:
        return {}

def extract_error_keywords(error_text):
    keywords = set()
    quoted = re.findall(r"['\"]([^'\"]+)['\"]", error_text)
    codes = re.findall(r"\[E\d{4}\]", error_text)
    tokens = re.findall(r"[A-Za-z][A-Za-z0-9]*", error_text)
    keywords.update(quoted)
    keywords.update(codes)
    keywords.update(tokens)
    return keywords

def build_resource_property_map(graph):
    resource_props = {}
    property_to_resources = {}

    for node_id, node_data in graph.nodes(data=True):
        if node_data.get("ntype") not in ("ResourceType", "Resource"):
            continue
        resource_props[node_id] = []
        for _, nbr in graph.out_edges(node_id):
            nbr_data = graph.nodes[nbr]
            if nbr_data.get("ntype") != "Property":
                continue
            prop_name = nbr_data.get("name", "")
            prop_info = {
                "name": prop_name,
                "type": nbr_data.get("type", "Any"),
                "primitive_type": nbr_data.get("primitive_type"),
                "required": nbr_data.get("required", False),
                "docs": nbr_data.get("docs", ""),
            }
            resource_props[node_id].append(prop_info)
            if prop_name not in property_to_resources:
                property_to_resources[prop_name] = []
            property_to_resources[prop_name].append(node_id)

    return resource_props, property_to_resources

def find_graph_matches(keywords, template_resources, resource_props, property_to_resources):
    matched = {}
    for kw in keywords:
        if kw in property_to_resources:
            for res in property_to_resources[kw]:
                if res not in matched:
                    matched[res] = {"score": 0, "matched_props": []}
                matched[res]["score"] += 10
                if kw not in matched[res]["matched_props"]:
                    matched[res]["matched_props"].append(kw)

        kw_lower = kw.lower()
        for res in resource_props:
            if kw_lower in res.lower():
                if res not in matched:
                    matched[res] = {"score": 0, "matched_props": []}
                matched[res]["score"] += 5

    for res in matched:
        if res in template_resources:
            matched[res]["score"] += 100

    return sorted(matched.items(), key=lambda x: -x[1]["score"])

def build_vector_rankings(query_results):
    best = {}
    if not (isinstance(query_results, dict) and query_results.get("metadatas") and query_results.get("distances")):
        return []
    metas = query_results["metadatas"][0]
    dists = query_results["distances"][0]
    for meta, dist in zip(metas, dists):
        label = meta.get("label", "") if isinstance(meta, dict) else str(meta)
        res = label.split("/")[0] if "/" in label else label
        if not res.startswith("AWS::"):
            continue
        dist = float(dist)
        if res not in best or dist < best[res]:
            best[res] = dist
    ranked = sorted(best.items(), key=lambda x: x[1])
    return [(res, {"distance": d, "score": 1.0 / (1.0 + d)}) for res, d in ranked]

def schema_snapshot(resource, resource_props, matched_props=None, max_optional=20):
    props = resource_props.get(resource, [])
    required = [p for p in props if p.get("required")]
    optional = [p for p in props if not p.get("required")]
    return {
        "resource": resource,
        "required_count": len(required),
        "optional_count": len(optional),
        "matched_props": matched_props or [],
        "required_props": [
            {"name": p.get("name", ""), "type": p.get("primitive_type") or p.get("type") or "Any"}
            for p in required
        ],
        "optional_props": [
            {"name": p.get("name", ""), "type": p.get("primitive_type") or p.get("type") or "Any"}
            for p in optional[:max_optional]
        ],
    }

def build_graphrag_rankings(graph, query_text, template_resources, resource_props, property_to_resources, seed_limit=15, max_hops=2):
    """Expand the strongest graph seeds into a local neighborhood subgraph."""
    seed_keywords = extract_error_keywords(query_text)
    seed_ranked = find_graph_matches(seed_keywords, template_resources, resource_props, property_to_resources)[:seed_limit]
    if not seed_ranked:
        return []

    undirected_graph = graph.to_undirected()
    candidate_scores = {}
    candidate_meta = {}

    for seed_rank, (seed_resource, seed_info) in enumerate(seed_ranked, start=1):
        if seed_resource not in graph:
            continue

        shortest_paths = nx.single_source_shortest_path_length(undirected_graph, seed_resource, cutoff=max_hops)
        for node_id, hop_distance in shortest_paths.items():
            node_type = graph.nodes[node_id].get("ntype")
            if node_type not in ("ResourceType", "Resource"):
                continue

            proximity_bonus = 25.0 / (hop_distance + 1)
            score = float(seed_info.get("score", 0)) + proximity_bonus
            if node_id in template_resources:
                score += 100

            if node_id not in candidate_scores or score > candidate_scores[node_id]:
                candidate_scores[node_id] = score
                candidate_meta[node_id] = {
                    "source_seed": seed_resource,
                    "seed_rank": seed_rank,
                    "hop_distance": hop_distance,
                    "matched_props": list(seed_info.get("matched_props", [])),
                    "seed_score": float(seed_info.get("score", 0)),
                }

    ranked = sorted(candidate_scores.items(), key=lambda item: -item[1])
    return [(resource, {**candidate_meta[resource], "score": score}) for resource, score in ranked]

resource_props, property_to_resources = build_resource_property_map(G)
print(f"Indexed resources: {len(resource_props)}")

Indexed resources: 1579


In [6]:
# !pip install neo4j

In [11]:
import re
import yaml
from neo4j import GraphDatabase

class CFNGraphRAG:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def _get_active_resources(self, template_yaml: str) -> set:
        """Parses the YAML to find AWS::X::Y resources currently in use."""
        try:
            tpl = yaml.safe_load(template_yaml)
            return {v.get("Type") for v in tpl.get("Resources", {}).values() if isinstance(v, dict)}
        except:
            return set()

    def _extract_flagged_properties(self, errors: list) -> set:
        """Extracts properties mentioned in single quotes by cfn-lint."""
        flagged = set()
        for error in errors:
            matches = re.findall(r"'([A-Za-z0-9]+)'", error)
            flagged.update(matches)
        return flagged

    def retrieve_schema_context(self, template_yaml: str, errors: list) -> str:
        active_resources = self._get_active_resources(template_yaml)
        flagged_props = self._extract_flagged_properties(errors)
        
        if not active_resources:
            return "No AWS resources identified in template."

        context_blocks = []
        with self.driver.session() as session:
            for resource in active_resources:
                # CYPHER QUERY: Fetch Resource, its properties, and nested types
                result = session.run("""
                    MATCH (r:Resource {name: $res_name})-[:HAS_PROPERTY]->(p:Property)
                    OPTIONAL MATCH (p)-[:USES_TYPE]->(pt:PropertyType)
                    RETURN p.name AS prop_name, p.type AS type, p.required AS required, pt.name AS nested_type
                """, res_name=resource)
                
                props = result.data()
                if not props:
                    continue

                required_props = [p['prop_name'] for p in props if p['required']]
                nested_types = [p['nested_type'] for p in props if p['nested_type']]
                
                # Filter flagged properties that actually belong to this resource
                valid_flagged = [p['prop_name'] for p in props if p['prop_name'] in flagged_props]
                invalid_flagged = [p for p in flagged_props if p not in [x['prop_name'] for x in props]]

                # Build Markdown Block
                block = f"### {resource}\n"
                
                if valid_flagged or invalid_flagged:
                    block += "**Properties flagged in errors:**\n"
                    for vp in valid_flagged:
                        match = next(p for p in props if p['prop_name'] == vp)
                        req = "required" if match['required'] else "optional"
                        block += f"  - `{vp}` ({match['type']}, {req})\n"
                    for ip in invalid_flagged:
                        block += f"  - `{ip}` *(invalid property name for this resource)*\n"

                block += f"**Required properties:** {', '.join(required_props)}\n"
                
                if nested_types:
                    # Strip the prefix for readability
                    clean_nested = [n.split('.')[-1] for n in set(nested_types)]
                    block += f"**Nested types:** {', '.join(clean_nested)}\n"

                context_blocks.append(block)

        return "\n".join(context_blocks)

# ==========================================
# Example Usage inside your Remediator Agent
# ==========================================

template = """
Resources:
  MyBucket:
    Type: AWS::S3::Bucket
    Properties:
      BucketName: my-test-bucket
      FakeProperty: True
"""
errors = ["[E3002] Additional properties are not allowed ('FakeProperty' was unexpected)"]

rag = CFNGraphRAG("bolt://localhost:7687", "neo4j", "password")
context = rag.retrieve_schema_context(template, errors)

print(context)

### AWS::S3::Bucket
**Properties flagged in errors:**
  - `FakeProperty` *(invalid property name for this resource)*
**Required properties:** 
**Nested types:** AnalyticsConfiguration, CorsConfiguration, ReplicationConfiguration, WebsiteConfiguration, NotificationConfiguration, LifecycleConfiguration, VersioningConfiguration, LoggingConfiguration, Tag, BucketEncryption, PublicAccessBlockConfiguration, MetadataConfiguration, IntelligentTieringConfiguration, OwnershipControls, AccelerateConfiguration, MetricsConfiguration, MetadataTableConfiguration, InventoryConfiguration, ObjectLockConfiguration



In [12]:
import chromadb
import uuid

# Build a temporary Chroma collection for vector search
chroma_client = chromadb.Client()
collection_name = "cfn_schema_debug"
try:
    chroma_client.delete_collection(name=collection_name)
except Exception:
    pass

collection = chroma_client.create_collection(name=collection_name)

sample_size = min(5000, len(corpus))
subset_corpus = corpus[:sample_size]
subset_metadata = [{"label": label} for label in corpus_labels[:sample_size]]
subset_ids = [str(uuid.uuid4()) for _ in range(sample_size)]

collection.add(
    documents=subset_corpus,
    metadatas=subset_metadata,
    ids=subset_ids,
    )

print(f"Chroma indexed documents: {sample_size}")

Chroma indexed documents: 5000


In [13]:
# ==============================
# Run Graph and Vector Retrieval
# ==============================
template_resource_map = parse_template_resources(selected_template)
template_resources = {v for v in template_resource_map.values() if v}

keywords = extract_error_keywords(query_text)
ranked_graph = find_graph_matches(
    keywords=keywords,
    template_resources=template_resources,
    resource_props=resource_props,
    property_to_resources=property_to_resources,
    )
ranked_graphrag = build_graphrag_rankings(
    graph=G,
    query_text=query_text,
    template_resources=template_resources,
    resource_props=resource_props,
    property_to_resources=property_to_resources,
    )

vector_results = collection.query(
    query_texts=[query_text],
    n_results=N_RESULTS_VECTOR,
    )
ranked_vector = build_vector_rankings(vector_results)

graph_rows = []
for idx, (res, info) in enumerate(ranked_graph, start=1):
    snap = schema_snapshot(res, resource_props, matched_props=info.get("matched_props", []), max_optional=MAX_OPTIONAL_SHOWN)
    graph_rows.append({
        "resource": res,
        "graph_rank": idx,
        "graph_score": info.get("score", 0),
        "required_props": snap["required_count"],
        "optional_props": snap["optional_count"],
        "matched_props": ", ".join(info.get("matched_props", [])[:8]),
    })

vector_rows = []
for idx, (res, info) in enumerate(ranked_vector, start=1):
    snap = schema_snapshot(res, resource_props, max_optional=MAX_OPTIONAL_SHOWN)
    vector_rows.append({
        "resource": res,
        "vector_rank": idx,
        "vector_distance": round(info.get("distance", 0.0), 6),
        "vector_score": round(info.get("score", 0.0), 6),
        "required_props": snap["required_count"],
        "optional_props": snap["optional_count"],
    })

graph_df = pd.DataFrame(graph_rows)
graphrag_rows = []
for idx, (res, info) in enumerate(ranked_graphrag, start=1):
    snap = schema_snapshot(res, resource_props, matched_props=info.get("matched_props", []), max_optional=MAX_OPTIONAL_SHOWN)
    graphrag_rows.append({
        "resource": res,
        "graphrag_rank": idx,
        "graphrag_score": info.get("score", 0),
        "source_seed": info.get("source_seed", ""),
        "hop_distance": info.get("hop_distance", ""),
        "required_props": snap["required_count"],
        "optional_props": snap["optional_count"],
        "matched_props": ", ".join(info.get("matched_props", [])[:8]),
    })
graphrag_df = pd.DataFrame(graphrag_rows)
vector_df = pd.DataFrame(vector_rows)

print(f"Keywords extracted: {len(keywords)}")
print(f"Graph matches: {len(graph_df)}")
print(f"GraphRAG matches: {len(graphrag_df)}")
print(f"Vector matches: {len(vector_df)}")

Keywords extracted: 9
Graph matches: 1579
GraphRAG matches: 15
Vector matches: 10


In [14]:
# ========================================
# Efficiency View + LLM Context Rendering
# ========================================
graph_top = graph_df.head(TOP_K_GRAPH).copy()
graphrag_top = graphrag_df.head(TOP_K_GRAPH).copy()
vector_top = vector_df.head(TOP_K_GRAPH).copy()

if not graph_top.empty and not vector_top.empty:
    comp_df = graph_top.merge(
        vector_top[["resource", "vector_rank", "vector_distance", "vector_score"]],
        on="resource",
        how="outer",
    )
elif not graph_top.empty:
    comp_df = graph_top.copy()
    comp_df["vector_rank"] = pd.NA
    comp_df["vector_distance"] = pd.NA
    comp_df["vector_score"] = pd.NA
else:
    comp_df = vector_top.copy()
    comp_df["graph_rank"] = pd.NA
    comp_df["graph_score"] = pd.NA

comp_df["in_both"] = comp_df["graph_rank"].notna() & comp_df["vector_rank"].notna()
comp_df = comp_df.sort_values(["graph_rank", "vector_rank"], na_position="last").reset_index(drop=True)

graph_resource_set = set(graph_top["resource"]) if not graph_top.empty else set()
vector_resource_set = set(vector_top["resource"]) if not vector_top.empty else set()
graphrag_resource_set = set(graphrag_top["resource"]) if not graphrag_top.empty else set()
overlap = graph_resource_set & vector_resource_set
graph_rag_overlap = graph_resource_set & graphrag_resource_set

print("=== Retrieval Efficiency Summary ===")
print(f"Run: {selected_run_id}")
print(f"Error text: {query_text}")
print(f"Top-K compared: {TOP_K_GRAPH}")
print(f"Graph candidates: {len(graph_df)}")
print(f"GraphRAG candidates: {len(graphrag_df)}")
print(f"Vector candidates: {len(vector_df)}")
print(f"Overlap in top-K: {len(overlap)}")
print(f"Graph vs GraphRAG overlap in top-K: {len(graph_rag_overlap)}")

if template_resources:
    graph_template_hits = len(graph_resource_set & template_resources)
    graphrag_template_hits = len(graphrag_resource_set & template_resources)
    vector_template_hits = len(vector_resource_set & template_resources)
    total_template = len(template_resources)
    print(f"Template resource coverage (Graph): {graph_template_hits}/{total_template}")
    print(f"Template resource coverage (GraphRAG): {graphrag_template_hits}/{total_template}")
    print(f"Template resource coverage (Vector): {vector_template_hits}/{total_template}")

display(comp_df)

print("\nTop GraphRAG-ranked resources:")
display(graphrag_top)

# Render schema context intended for LLM remediation input
graph_context = []
for _, row in graph_df.head(TOP_K_CONTEXT).iterrows():
    matched_props = [p.strip() for p in str(row.get("matched_props", "")).split(",") if p.strip()]
    snap = schema_snapshot(
        resource=row["resource"],
        resource_props=resource_props,
        matched_props=matched_props,
        max_optional=MAX_OPTIONAL_SHOWN,
    )
    snap["rank"] = int(row["graph_rank"])
    snap["score"] = float(row["graph_score"])
    graph_context.append(snap)

graphrag_context = []
for _, row in graphrag_df.head(TOP_K_CONTEXT).iterrows():
    matched_props = [p.strip() for p in str(row.get("matched_props", "")).split(",") if p.strip()]
    snap = schema_snapshot(
        resource=row["resource"],
        resource_props=resource_props,
        matched_props=matched_props,
        max_optional=MAX_OPTIONAL_SHOWN,
    )
    snap["rank"] = int(row["graphrag_rank"])
    snap["score"] = float(row["graphrag_score"])
    snap["source_seed"] = row.get("source_seed", "")
    snap["hop_distance"] = int(row.get("hop_distance", 0))
    graphrag_context.append(snap)

vector_context = []
for _, row in vector_df.head(TOP_K_CONTEXT).iterrows():
    snap = schema_snapshot(
        resource=row["resource"],
        resource_props=resource_props,
        matched_props=[],
        max_optional=MAX_OPTIONAL_SHOWN,
    )
    snap["rank"] = int(row["vector_rank"])
    snap["distance"] = float(row["vector_distance"])
    snap["score"] = float(row["vector_score"])
    vector_context.append(snap)

llm_context_payload = {
    "query_mode": QUERY_MODE,
    "run_id": selected_run_id,
    "query_text": query_text,
    "template_resources": sorted(list(template_resources)),
    "graph_context_topk": graph_context,
    "graphrag_context_topk": graphrag_context,
    "vector_context_topk": vector_context,
}

print("\n=== LLM Context Payload (preview) ===")
print(json.dumps({
    "run_id": llm_context_payload["run_id"],
    "query_mode": llm_context_payload["query_mode"],
    "template_resources": llm_context_payload["template_resources"],
    "graph_context_topk_count": len(llm_context_payload["graph_context_topk"]),
    "graphrag_context_topk_count": len(llm_context_payload["graphrag_context_topk"]),
    "vector_context_topk_count": len(llm_context_payload["vector_context_topk"]),
}, indent=2))

print("\nGraph context sample (first resource):")
if graph_context:
    print(json.dumps(graph_context[0], indent=2))
else:
    print("No graph context available")

print("\nGraphRAG context sample (first resource):")
if graphrag_context:
    print(json.dumps(graphrag_context[0], indent=2))
else:
    print("No GraphRAG context available")

print("\nVector context sample (first resource):")
if vector_context:
    print(json.dumps(vector_context[0], indent=2))
else:
    print("No vector context available")

=== Retrieval Efficiency Summary ===
Run: 20260412_014839_fab97a2b
Error text: [E3003] {'ColumnNumber': 7, 'LineNumber': 20}: 'SSEEnabled' is a required property
Top-K compared: 20
Graph candidates: 1579
GraphRAG candidates: 15
Vector candidates: 10
Overlap in top-K: 0
Graph vs GraphRAG overlap in top-K: 15
Template resource coverage (Graph): 1/1
Template resource coverage (GraphRAG): 1/1
Template resource coverage (Vector): 0/1


,resource,graph_rank,graph_score,required_props,optional_props,matched_props,vector_rank,vector_distance,vector_score,in_both
0,AWS::DynamoDB::Table,1.0,105.0,1.0,19.0,,NaN,NaN,NaN,False
1,AWS::ACMPCA::Permission,2.0,10.0,3.0,1.0,,NaN,NaN,NaN,False
2,AWS::ApplicationSignals::Discovery,3.0,10.0,0.0,0.0,,NaN,NaN,NaN,False
3,AWS::BackupGateway::Hypervisor,4.0,10.0,0.0,7.0,,NaN,NaN,NaN,False
4,AWS::CleanRooms::AnalysisTemplate,5.0,10.0,4.0,7.0,,NaN,NaN,NaN,False
5,AWS::CloudFormation::Publisher,6.0,10.0,1.0,1.0,,NaN,NaN,NaN,False
6,AWS::CloudFront::AnycastIpList,7.0,10.0,2.0,3.0,,NaN,NaN,NaN,False
7,AWS::CloudFront::Distribution,8.0,10.0,1.0,1.0,,NaN,NaN,NaN,False
8,AWS::CloudFront::DistributionTenant,9.0,10.0,3.0,6.0,,NaN,NaN,NaN,False
9,AWS::CloudFront::StreamingDistribution,10.0,10.0,2.0,0.0,,NaN,NaN,NaN,False



Top GraphRAG-ranked resources:


,resource,graphrag_rank,graphrag_score,source_seed,hop_distance,required_props,optional_props,matched_props
0,AWS::DynamoDB::Table,1,230.0,AWS::DynamoDB::Table,0,1,19,
1,AWS::ACMPCA::Permission,2,35.0,AWS::ACMPCA::Permission,0,3,1,
2,AWS::ApplicationSignals::Discovery,3,35.0,AWS::ApplicationSignals::Discovery,0,0,0,
3,AWS::BackupGateway::Hypervisor,4,35.0,AWS::BackupGateway::Hypervisor,0,0,7,
4,AWS::CleanRooms::AnalysisTemplate,5,35.0,AWS::CleanRooms::AnalysisTemplate,0,4,7,
5,AWS::CloudFormation::Publisher,6,35.0,AWS::CloudFormation::Publisher,0,1,1,
6,AWS::CloudFront::AnycastIpList,7,35.0,AWS::CloudFront::AnycastIpList,0,2,3,
7,AWS::CloudFront::Distribution,8,35.0,AWS::CloudFront::Distribution,0,1,1,
8,AWS::CloudFront::DistributionTenant,9,35.0,AWS::CloudFront::DistributionTenant,0,3,6,
9,AWS::CloudFront::StreamingDistribution,10,35.0,AWS::CloudFront::StreamingDistribution,0,2,0,



=== LLM Context Payload (preview) ===
{
  "run_id": "20260412_014839_fab97a2b",
  "query_mode": "from_csv",
  "template_resources": [
    "AWS::DynamoDB::Table"
  ],
  "graph_context_topk_count": 5,
  "graphrag_context_topk_count": 5,
  "vector_context_topk_count": 5
}

Graph context sample (first resource):
{
  "resource": "AWS::DynamoDB::Table",
  "required_count": 1,
  "optional_count": 19,
  "matched_props": [],
  "required_props": [
    {
      "name": "KeySchema",
      "type": "List"
    }
  ],
  "optional_props": [
    {
      "name": "OnDemandThroughput",
      "type": "OnDemandThroughput"
    },
    {
      "name": "SSESpecification",
      "type": "SSESpecification"
    },
    {
      "name": "KinesisStreamSpecification",
      "type": "KinesisStreamSpecification"
    },
    {
      "name": "StreamSpecification",
      "type": "StreamSpecification"
    },
    {
      "name": "ContributorInsightsSpecification",
      "type": "ContributorInsightsSpecification"
    },
    {
  

In [19]:
print(selected_template)

AWSTemplateFormatVersion: '2010-09-09'
Description: DynamoDB table for ArtistId, Concert, and TicketSales data with ArtistId as partition key and Concert as sort key for efficient queries.

Resources:
  ArtistConcertTicketSalesTable:
    Type: AWS::DynamoDB::Table
    Properties:
      TableName: !Sub "${AWS::StackName}-ArtistConcertTicketSalesTable-${AWS::Region}-${AWS::AccountId}"
      AttributeDefinitions:
        - AttributeName: ArtistId
          AttributeType: S
        - AttributeName: Concert
          AttributeType: S
      KeySchema:
        - AttributeName: ArtistId
          KeyType: HASH
        - AttributeName: Concert
          KeyType: RANGE
      BillingMode: PAY_PER_REQUEST
      SSESpecification:
        SSEEnabled: true
        SSEType: KMS
        KMSMasterKeyId: alias/aws/dynamodb
      PointInTimeRecoverySpecification:
        PointInTimeRecoveryEnabled: true
      DeletionProtectionEnabled: true
      ContributorInsightsSpecification:
        Enabled: false
  

# Neo4j GraphRAG Trial

This trial uses a Neo4j-backed graph lookup to build schema context from the active template and its validation errors. It is intended as a final RAG approach alongside the heuristic graph search, GraphRAG expansion, and vector search cells above.

In [20]:
import re
import yaml

try:
    from neo4j import GraphDatabase
except ImportError as exc:
    GraphDatabase = None
    neo4j_import_error = exc

if GraphDatabase is None:
    print(f"Neo4j trial skipped: {neo4j_import_error}")
else:
    class CFNGraphRAGNeo4j:
        def __init__(self, uri, user, password):
            self.driver = GraphDatabase.driver(uri, auth=(user, password))

        def _get_active_resources(self, template_yaml: str) -> set:
            """Parse the YAML and collect AWS resource types used in the template."""
            try:
                tpl = yaml.safe_load(template_yaml) or {}
                return {v.get("Type") for v in tpl.get("Resources", {}).values() if isinstance(v, dict)}
            except Exception:
                return set()

        def _extract_flagged_properties(self, errors: list) -> set:
            """Extract property names enclosed in single quotes from validation errors."""
            flagged = set()
            for error in errors:
                matches = re.findall(r"'([A-Za-z0-9]+)'", error)
                flagged.update(matches)
            return flagged

        def _fetch_property_type_properties(self, session, type_name: str) -> list:
            """Fetch properties defined on a PropertyType node."""
            result = session.run("""
                MATCH (pt:PropertyType {name: $type_name})-[:HAS_PROPERTY]->(p:Property)
                OPTIONAL MATCH (p)-[:USES_TYPE]->(npt:PropertyType)
                RETURN p.name AS prop_name, p.type AS type, p.required AS required, npt.name AS nested_type
            """, type_name=type_name)
            return result.data()

        def retrieve_schema_context(self, template_yaml: str, errors: list) -> str:
            active_resources = self._get_active_resources(template_yaml)
            flagged_props = self._extract_flagged_properties(errors)

            if not active_resources:
                return "No AWS resources identified in template."

            context_blocks = []
            with self.driver.session() as session:
                for resource in sorted(active_resources):
                    direct_result = session.run("""
                        MATCH (r:Resource {name: $res_name})-[:HAS_PROPERTY]->(p:Property)
                        OPTIONAL MATCH (p)-[:USES_TYPE]->(pt:PropertyType)
                        RETURN p.name AS prop_name, p.type AS type, p.required AS required, pt.name AS nested_type
                    """, res_name=resource)

                    direct_props = direct_result.data()
                    if not direct_props:
                        continue

                    nested_type_names = sorted({row["nested_type"] for row in direct_props if row.get("nested_type")})
                    nested_props_by_type = {
                        type_name: self._fetch_property_type_properties(session, type_name)
                        for type_name in nested_type_names
                    }

                    direct_required = [row["prop_name"] for row in direct_props if row.get("required")]
                    direct_flagged = [row for row in direct_props if row["prop_name"] in flagged_props]

                    nested_flagged = []
                    for type_name, props in nested_props_by_type.items():
                        for prop in props:
                            if prop["prop_name"] in flagged_props:
                                nested_flagged.append((type_name, prop))

                    block_lines = [f"### {resource}"]
                    if direct_flagged or nested_flagged:
                        block_lines.append("**Properties flagged in errors:**")
                        for row in direct_flagged:
                            req = "required" if row["required"] else "optional"
                            block_lines.append(f"- `{row['prop_name']}` ({row['type']}, {req})")
                        for type_name, prop in nested_flagged:
                            req = "required" if prop["required"] else "optional"
                            block_lines.append(f"- `{prop['prop_name']}` ({prop['type']}, {req}) under `{type_name}`")

                    block_lines.append(f"**Required properties:** {', '.join(direct_required) if direct_required else '(none)'}")
                    if nested_type_names:
                        block_lines.append(f"**Nested property types:** {', '.join(nested_type_names)}")
                    for type_name in nested_type_names:
                        nested_props = nested_props_by_type.get(type_name, [])
                        if not nested_props:
                            continue
                        block_lines.append(f"**{type_name} properties:**")
                        for prop in nested_props:
                            req = "required" if prop["required"] else "optional"
                            marker = " *flagged*" if prop["prop_name"] in flagged_props else ""
                            block_lines.append(f"- `{prop['prop_name']}` ({prop['type']}, {req}){marker}")

                    context_blocks.append("\n".join(block_lines))

            return "\n\n".join(context_blocks) if context_blocks else "No graph schema context returned from Neo4j."

    # Example trial input: reuse the current notebook-selected template and error text if available.
    trial_template = selected_template if 'selected_template' in globals() else ''
    trial_errors = [query_text] if 'query_text' in globals() and query_text else []

    print("=== Neo4j GraphRAG Trial ===")
    print("This trial now retrieves nested properties from PropertyType nodes as well.")

    neo4j_uri = "bolt://localhost:7687"
    neo4j_user = "neo4j"
    neo4j_password = "password"

    try:
        rag_neo4j = CFNGraphRAGNeo4j(neo4j_uri, neo4j_user, neo4j_password)
        neo4j_context = rag_neo4j.retrieve_schema_context(trial_template, trial_errors)
        print(neo4j_context)
    except Exception as exc:
        print(f"Neo4j trial could not run: {exc}")

=== Neo4j GraphRAG Trial ===
This trial now retrieves nested properties from PropertyType nodes as well.
### AWS::DynamoDB::Table
**Properties flagged in errors:**
- `SSEEnabled` (Boolean, required) under `AWS::DynamoDB::Table.SSESpecification`
**Required properties:** KeySchema
**Nested property types:** AWS::DynamoDB::Table.AttributeDefinition, AWS::DynamoDB::Table.ContributorInsightsSpecification, AWS::DynamoDB::Table.GlobalSecondaryIndex, AWS::DynamoDB::Table.ImportSourceSpecification, AWS::DynamoDB::Table.KeySchema, AWS::DynamoDB::Table.KinesisStreamSpecification, AWS::DynamoDB::Table.LocalSecondaryIndex, AWS::DynamoDB::Table.OnDemandThroughput, AWS::DynamoDB::Table.PointInTimeRecoverySpecification, AWS::DynamoDB::Table.ProvisionedThroughput, AWS::DynamoDB::Table.ResourcePolicy, AWS::DynamoDB::Table.SSESpecification, AWS::DynamoDB::Table.StreamSpecification, AWS::DynamoDB::Table.Tag, AWS::DynamoDB::Table.TimeToLiveSpecification, AWS::DynamoDB::Table.WarmThroughput
**AWS::DynamoDB: